## Setup

In [1]:
import sys
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Import app functions
from app.gradio_app import (
    get_available_models,
    load_model,
    preprocess_image,
    predict_latex,
    render_latex_with_sympy,
    TOKENIZER,
    DEVICE
)

print(f"Device: {DEVICE}")
print(f"Vocabulary size: {TOKENIZER.vocab_size}")

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INFO] Using device: cuda
[INFO] Checkpoint directory: /app/checkpoints
[INFO] Vocab path: /app/vocab.json
[INFO] Found 8 model checkpoints
[INFO] Vocabulary size: 500
Device: cuda
Vocabulary size: 500


## 1. Discover Available Models

In [2]:
models = get_available_models()

print(f"Found {len(models)} model checkpoints:\n")
for name, path in models.items():
    print(f"  • {name}")
    print(f"    {path}\n")

Found 8 model checkpoints:

  • model_a/baseline/best
    /app/checkpoints/model_a/baseline/best.pt

  • model_a/finetuned/best
    /app/checkpoints/model_a/finetuned/best.pt

  • model_b/baseline/best
    /app/checkpoints/model_b/baseline/best.pt

  • model_b/finetuned/best
    /app/checkpoints/model_b/finetuned/best.pt

  • model_c/baseline/best
    /app/checkpoints/model_c/baseline/best.pt

  • model_c/finetuned/best
    /app/checkpoints/model_c/finetuned/best.pt

  • model_d/baseline/best
    /app/checkpoints/model_d/baseline/best.pt

  • model_d/finetuned/best
    /app/checkpoints/model_d/finetuned/best.pt



## 2. Load a Model

In [3]:
# Select first available model
if models:
    model_name = list(models.keys())[0]
    model_path = models[model_name]
    
    print(f"Loading: {model_name}")
    model = load_model(model_path, TOKENIZER.vocab_size)
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params:,}")
else:
    print("No models found! Please train a model first.")
    model = None

Loading: model_a/baseline/best
[INFO] Detected model type: model_a
[INFO] Inferred hidden_dim=256
[INFO] Loading Model A (CNN + BiLSTM + CTC)
[OK] Loaded model_a from /app/checkpoints/model_a/baseline/best.pt
Total parameters: 13,420,148


## 3. Create Sample Image

For testing, let's create a simple synthetic image.

In [4]:
# Create a sample image (white background with black text pattern)
width, height = 300, 100
sample_image = np.ones((height, width, 3), dtype=np.uint8) * 255

# Draw some "text-like" patterns
sample_image[30:50, 50:80] = 0      # Block 1
sample_image[20:40, 100:110] = 0    # Block 2 (superscript-like)
sample_image[35:45, 130:150] = 0    # Block 3
sample_image[30:50, 170:200] = 0    # Block 4

plt.figure(figsize=(8, 3))
plt.imshow(sample_image)
plt.title("Sample Input Image")
plt.axis('off')
plt.show()

print(f"Image shape: {sample_image.shape}")

Image shape: (100, 300, 3)


## 4. Preprocess Image

In [5]:
# Preprocess with IM2LATEX pipeline
preprocessed = preprocess_image(sample_image, "IM2LATEX")

print(f"Preprocessed shape: {preprocessed.shape}")
print(f"Expected: (128, 768)")

# Visualize
plt.figure(figsize=(12, 3))
plt.imshow(preprocessed, cmap='gray')
plt.title("Preprocessed Image (128x768)")
plt.axis('off')
plt.show()

Preprocessed shape: (128, 768)
Expected: (128, 768)


## 5. Predict LaTeX

In [6]:
if model is not None:
    # Predict
    latex_output = predict_latex(model, TOKENIZER, preprocessed, max_len=200)
    
    print("="*60)
    print("PREDICTED LATEX:")
    print("="*60)
    print(latex_output)
    print("="*60)
else:
    print("Model not loaded. Skipping prediction.")

PREDICTED LATEX:
\hspace \qquad \bf \bullet \bf \bf \bullet \, \bf a } {


## 6. Render LaTeX with SymPy

In [7]:
if model is not None:
    # Render LaTeX
    rendered = render_latex_with_sympy(latex_output)
    
    if rendered is not None:
        plt.figure(figsize=(10, 3))
        plt.imshow(rendered)
        plt.title("Rendered LaTeX (SymPy)")
        plt.axis('off')
        plt.show()
        
        print(f"Rendered image size: {rendered.size}")
    else:
        print("SymPy rendering not available or failed.")
        print("You can still use the LaTeX output in other editors.")

[WARNING] Failed to render LaTeX with SymPy: 
\hspace \qquad \bf \bullet \bf \bf \bullet \, \bf a
        ^
ParseSyntaxException: Expected \hspace{space}, found '\'  (at char 8), (line:1, col:9)
[DEBUG] Original LaTeX: \hspace \qquad \bf \bullet \bf \bf \bullet \, \bf a } {...
[DEBUG] Cleaned LaTeX: \hspace \qquad \bf \bullet \bf \bf \bullet \, \bf a...
SymPy rendering not available or failed.
You can still use the LaTeX output in other editors.


## 7. Test with Real Images

If you have test images, try them here!

In [8]:
# Example: Load a real test image
test_image_path = Path("../data/CROHME/CROHME_test_2011/algb02.png")

if test_image_path.exists() and model is not None:
    # Load image
    test_img = np.array(Image.open(test_image_path))
    
    # Show original
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(test_img, cmap='gray' if len(test_img.shape) == 2 else None)
    plt.title("Original Image")
    plt.axis('off')
    
    # Preprocess
    preprocessed_test = preprocess_image(test_img, "CROHME")
    
    plt.subplot(1, 2, 2)
    plt.imshow(preprocessed_test, cmap='gray')
    plt.title("Preprocessed")
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    # Predict
    latex_result = predict_latex(model, TOKENIZER, preprocessed_test)
    print(f"\nPredicted LaTeX: {latex_result}")
    
    # Render
    rendered_result = render_latex_with_sympy(latex_result)
    if rendered_result:
        plt.figure(figsize=(8, 2))
        plt.imshow(rendered_result)
        plt.title("Rendered Result")
        plt.axis('off')
        plt.show()
else:
    print(f"Test image not found at: {test_image_path}")
    print("Place your test images and update the path above.")

Test image not found at: ../data/CROHME/CROHME_test_2011/algb02.png
Place your test images and update the path above.


## 8. Batch Processing Example

In [9]:
# Example: Process multiple images
test_dir = Path("../data/CROHME/CROHME_test_2011")

if test_dir.exists() and model is not None:
    # Get first 5 test images
    test_images = list(test_dir.glob("*.png"))[:5]
    
    print(f"Processing {len(test_images)} images...\n")
    
    results = []
    for img_path in test_images:
        try:
            # Load and preprocess
            img = np.array(Image.open(img_path))
            preprocessed = preprocess_image(img, "CROHME")
            
            # Predict
            latex = predict_latex(model, TOKENIZER, preprocessed)
            
            results.append({
                'filename': img_path.name,
                'latex': latex
            })
            
            print(f"✓ {img_path.name}: {latex}")
        
        except Exception as e:
            print(f"✗ {img_path.name}: Error - {e}")
    
    print(f"\nProcessed {len(results)} images successfully!")
else:
    print("Test directory not found or model not loaded.")

Processing 0 images...


Processed 0 images successfully!
